# 04 — Customer Segmentation via RFM Clustering

**Technique:** K-Means clustering on RFM features
**Goal:** Identify distinct customer segments for targeted marketing.

**RFM Framework:**
| Metric | Definition |
|--------|-----------|
| **Recency** | Days since last purchase (lower = more recent) |
| **Frequency** | Number of unique invoices |
| **Monetary** | Total spend (£) |

**Pipeline:** Build RFM → Cap outliers → Log-transform → Scale → Elbow method → K-Means → Label segments


In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
os.makedirs('outputs/figures', exist_ok=True)
print(f'Working dir: {os.getcwd()}')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
%matplotlib inline
sns.set_theme(style='whitegrid')
SAVE = lambda name: plt.savefig(f'outputs/figures/{name}', dpi=150, bbox_inches='tight')

## 1. Build RFM Table

In [ ]:
df = pd.read_csv('data/cleaned_retail_customers.csv', parse_dates=['InvoiceDate'])
snapshot = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot - x.max()).days),
    Frequency = ('Invoice',     'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

print(f"RFM table: {rfm.shape}")
rfm.describe().round(2)

## 2. Handle Outliers & Scale

In [ ]:
# Cap at 99th percentile to reduce outlier influence
for col in ['Recency','Frequency','Monetary']:
    cap = rfm[col].quantile(0.99)
    rfm[col] = rfm[col].clip(upper=cap)

# Log-transform right-skewed columns
rfm['Freq_log'] = np.log1p(rfm['Frequency'])
rfm['Mon_log']  = np.log1p(rfm['Monetary'])

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency','Freq_log','Mon_log']])
print("RFM features scaled. Preview:")
pd.DataFrame(rfm_scaled, columns=['Recency_sc','Freq_sc','Mon_sc']).describe().round(3)

## 3. Elbow Method — Choose Optimal k

In [ ]:
inertias, silhouettes = [], []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(rfm_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=7)
ax1.set_title('Elbow Method (Inertia)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax2.plot(K_range, silhouettes, 'rs-', linewidth=2, markersize=7)
ax2.axvline(x=4, color='gray', linestyle='--', alpha=0.6, label='k=4 selected')
ax2.set_title('Silhouette Score', fontsize=13, fontweight='bold')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.legend()
plt.suptitle('Optimal Number of Clusters', fontsize=14, fontweight='bold')
plt.tight_layout()
SAVE('clustering_elbow.png')
plt.show()

## 4. K-Means Clustering (k = 4)

In [ ]:
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=10)
rfm['Cluster'] = km.fit_predict(rfm_scaled)

# Rank clusters by mean Monetary value → assign business labels
profile = (rfm.groupby('Cluster')
              .agg(Count=('CustomerID','count'),
                   Recency_avg=('Recency','mean'),
                   Frequency_avg=('Frequency','mean'),
                   Monetary_avg=('Monetary','mean'))
              .sort_values('Monetary_avg', ascending=False)
              .reset_index())

SEG_NAMES = ['Champions','Loyal Customers','At Risk','Lost / Inactive']
seg_map = {row['Cluster']: SEG_NAMES[i] for i, row in profile.iterrows()}
rfm['Segment'] = rfm['Cluster'].map(seg_map)

print("Cluster profiles (sorted by avg spend):")
profile['Segment'] = [SEG_NAMES[i] for i in range(K)]
print(profile.to_string(index=False))

## 5. Visualize Clusters

In [ ]:
palette = {'Champions':'#E74C3C','Loyal Customers':'#2ECC71',
             'At Risk':'#F39C12','Lost / Inactive':'#95A5A6'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for seg, color in palette.items():
    mask = rfm['Segment'] == seg
    ax1.scatter(rfm[mask]['Recency'], rfm[mask]['Mon_log'],
                label=seg, alpha=0.45, s=18, color=color)
ax1.set_title('Recency vs Log(Monetary)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Recency (days since last purchase)')
ax1.set_ylabel('log(Monetary)')
ax1.legend(fontsize=9)

for seg, color in palette.items():
    mask = rfm['Segment'] == seg
    ax2.scatter(rfm[mask]['Frequency'], rfm[mask]['Mon_log'],
                label=seg, alpha=0.45, s=18, color=color)
ax2.set_title('Frequency vs Log(Monetary)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Frequency (# unique invoices)')
ax2.set_ylabel('log(Monetary)')
ax2.legend(fontsize=9)
plt.tight_layout()
SAVE('clustering_scatter.png')
plt.show()

In [ ]:
# Segment size bar chart
seg_counts = rfm['Segment'].value_counts().reindex(SEG_NAMES)
colors_list = [palette[s] for s in seg_counts.index]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(seg_counts.index, seg_counts.values, color=colors_list, edgecolor='white', linewidth=1.5)
for bar, cnt in zip(bars, seg_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+8,
            f'{cnt:,}\n({cnt/len(rfm)*100:.0f}%)', ha='center', fontsize=10)
ax.set_title('Customer Segment Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Customers')
plt.tight_layout()
SAVE('clustering_segments.png')
plt.show()

In [ ]:
# RFM heatmap
profile_heat = rfm.groupby('Segment')[['Recency','Frequency','Monetary']].mean().reindex(SEG_NAMES)
profile_norm = (profile_heat - profile_heat.min()) / (profile_heat.max() - profile_heat.min())

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(profile_norm, annot=profile_heat.round(0), fmt='.0f', cmap='RdYlGn',
            ax=ax, linewidths=0.5, cbar_kws={'label': 'Normalized (0–1)'})
ax.set_title('RFM Segment Profiles (raw values annotated)', fontsize=12, fontweight='bold')
plt.tight_layout()
SAVE('clustering_profiles.png')
plt.show()

rfm.to_csv('data/rfm_clustered.csv', index=False)
print("Saved data/rfm_clustered.csv")
print("Clustering complete!")